# SARIMAX Test Pipeline
This notebook adapts the AutoARIMA pipeline for SARIMAX, using the same recursive inference and validation logic.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os

In [2]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)

# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        df = df.reset_index(drop=True)
    else:
        df = df.reset_index()
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
df = df.reset_index(drop=True)

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True)

y = df[TARGET_COL]
print(len(y))

# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
train_size = 455
val_size = 153
forecast_horizon = 153
lookback_window = 30
df.head()

Loading data from ../dataset/subset_set.feather...
7610


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_value_DISC,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id
0,2021-01-23,26008,104,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
1,2021-01-23,921558,15,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
2,2021-01-23,213626,85,milknplant based bevs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
3,2021-01-23,213625,44,milknplant based bevs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269
4,2021-01-23,213624,40,milknplant based bevs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,6269


In [3]:
import numpy as np
import pandas as pd
from pmdarima import auto_arima

def generate_autoarima_candidates(
    df_product: pd.DataFrame,
    date_col: str,
    target_col: str,
    exog_cols=None,
    train_size: int = 432,
    val_size: int = 153,
    forecast_window: int = 153,
    seasonal_periods=(0, 7),          # 0 = non-seasonal; 7 typical for daily retail
    top_k: int = 5,                   # how many candidates per m
    info_criterion: str = "aicc",     # "aic", "bic", "aicc"
    max_p: int = 5,
    max_q: int = 5,
    max_d: int = 2,
    max_P: int = 2,
    max_Q: int = 2,
    max_D: int = 1,
    stepwise: bool = True,
    n_random_fits: int = 50,
    random_state: int = 42,
):
    """
    Returns a dataframe of candidate (order, seasonal_order) models suggested by AutoARIMA.

    Approach:
    - Fit AutoARIMA for each seasonal period m in `seasonal_periods`
    - For each m, also run a randomized search (optional) to get extra candidates
    - Collect unique candidates and rank by information criterion
    """
    dfp = df_product.copy()
    dfp[date_col] = pd.to_datetime(dfp[date_col])
    dfp = dfp.sort_values(date_col).reset_index(drop=True)

    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val + forecast_window), -forecast_window)

    y = dfp.loc[train_slice, target_col].astype(float).values

    X = None
    if exog_cols:
        X = dfp.loc[train_slice, exog_cols].astype(float).values

    candidates = []

    def _record_candidate(model, m):
        order = tuple(model.order)
        sorder = tuple(model.seasonal_order) if m and getattr(model, "seasonal_order", None) else (0, 0, 0, 0)
        # Some versions store criterion under different attrs; aic() is usually available
        aic  = getattr(model, "aic", lambda: np.nan)()
        bic  = getattr(model, "bic", lambda: np.nan)()
        # pmdarima doesn't always expose aicc(); we can approximate if needed via model.arima_res_.aicc
        aicc = np.nan
        try:
            aicc = model.arima_res_.aicc
        except Exception:
            pass

        ic_map = {"aic": aic, "bic": bic, "aicc": aicc}
        ic_val = ic_map.get(info_criterion.lower(), aicc)

        candidates.append({
            "m": int(m),
            "order": order,
            "seasonal_order": sorder,
            "aic": aic,
            "bic": bic,
            "aicc": aicc,
            "ic": ic_val
        })

    for m in seasonal_periods:
        seasonal = (m is not None) and (int(m) > 0)

        # 1) Stepwise (deterministic) candidate
        try:
            model = auto_arima(
                y, X=X,
                seasonal=seasonal, m=int(m) if seasonal else 0,
                stepwise=stepwise,
                suppress_warnings=True,
                error_action="ignore",
                information_criterion=info_criterion,
                max_p=max_p, max_q=max_q, max_d=max_d,
                max_P=max_P, max_Q=max_Q, max_D=max_D,
                with_intercept=True,
            )
            _record_candidate(model, m)
        except Exception as e:
            print(f"[WARN] stepwise auto_arima failed for m={m}: {e}")

        # 2) Random search to harvest more candidate orders (optional but useful)
        #    This is the only part where random_state matters.
        try:
            model_rand = auto_arima(
                y, X=X,
                seasonal=seasonal, m=int(m) if seasonal else 0,
                stepwise=False,
                random=True,
                n_fits=n_random_fits,
                random_state=random_state,
                suppress_warnings=True,
                error_action="ignore",
                information_criterion=info_criterion,
                max_p=max_p, max_q=max_q, max_d=max_d,
                max_P=max_P, max_Q=max_Q, max_D=max_D,
                with_intercept=True,
            )
            _record_candidate(model_rand, m)
        except Exception as e:
            print(f"[WARN] random auto_arima failed for m={m}: {e}")

    cand_df = pd.DataFrame(candidates)

    # Deduplicate by (order, seasonal_order, m) then rank by chosen IC
    cand_df = (
        cand_df.sort_values("ic", na_position="last")
               .drop_duplicates(subset=["m", "order", "seasonal_order"], keep="first")
               .reset_index(drop=True)
    )

    # Return top_k per m
    out = (
        cand_df.groupby("m", group_keys=False)
               .apply(lambda g: g.nsmallest(top_k, "ic"))
               .reset_index(drop=True)
    )
    return out


count    7610.000000
mean       50.661761
std        30.226802
min         0.000000
25%        29.000000
50%        48.000000
75%        66.000000
max       504.000000
Name: value, dtype: float64


In [4]:

# -----------------------------------------------------------------------------
# FEATURE ENGINEERING (EXOGENOUS VARIABLES)
# -----------------------------------------------------------------------------
# 1. Day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df[DATE_COL].dt.dayofweek

# 2. Month (1=January, 12=December)
df['month'] = df[DATE_COL].dt.month

# 3. Day of month (1-31) - captures paydays
df['day_of_month'] = df[DATE_COL].dt.day

# 4. Is Weekend (Binary)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)

df['is_christmas'] = ((df['month'] == 12) & (df['day_of_month'] == 25)).astype(float)
# 5. Promotions
# Identify all columns that start with 'promo_'
promo_cols = [col for col in df.columns if col.startswith('promo_')]
print(f"Found {len(promo_cols)} promotion columns: {promo_cols}")

# Ensure promo columns are numeric (float)
for col in promo_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)

# Define the list of exogenous features to use
EXOG_COLS = ['day_of_week', 'month', 'day_of_month', 'is_weekend'] + promo_cols

Found 16 promotion columns: ['promo_type_FRPG', 'promo_value_FRPG', 'promo_type_GAS', 'promo_value_GAS', 'promo_type_BOGO', 'promo_value_BOGO', 'promo_type_DISC', 'promo_value_DISC', 'promo_type_CIRC', 'promo_value_CIRC', 'promo_type_CIRE', 'promo_value_CIRE', 'promo_type_CLCP', 'promo_value_CLCP', 'promo_type_LFPE', 'promo_value_LFPE']


# SARIMAX Experiment Function
This function matches the AutoARIMA pipeline: same splits, recursive inference, and validation.

In [6]:
def sarimax_recursive_forecast(df, target, exog_cols,  item_id, store_id, train_size=432, val_size=153, forecast_window=153, save_plot_path=None, order=(1,1,1), seasonal_order=(0,0,0,0)):
    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val+forecast_window), -forecast_window)
    test_slice = slice(-forecast_window, None)
    train = df[target][train_slice].values.tolist()
    test = df[target][test_slice].values
    exog = df[exog_cols]
    exog_train = exog[train_slice].values.tolist()
    exog_test = exog[test_slice].values
    forecast = []
    current_train = train.copy()
    current_exog = exog_train.copy()
    for i in range(forecast_window):
        # Fit SARIMAX on current window
        model = SARIMAX(current_train, exog=np.array(current_exog), order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
        results = model.fit(disp=False)
        # Forecast one step ahead
        next_exog = exog_test[i].reshape(1, -1)
        next_pred = results.forecast(steps=1, exog=next_exog)[0]
        forecast.append(next_pred)
        # Append actual value for next fit (simulate real scenario)
        current_train.append(test[i])
        current_exog.append(exog_test[i])
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    if save_plot_path:
        plt.figure(figsize=(12,6))
        plt.plot(range(len(train)), train, label='Train')
        plt.plot(range(len(train), len(train)+len(test)), test, label='Test')
        plt.plot(range(len(train), len(train)+len(test)), forecast, label='Forecast')
        plt.title(f'SARIMAX Forecast (Item={item_id}, Store={store_id})')
        plt.legend()
        plt.savefig(save_plot_path)
        plt.close()
    return forecast, rmse, mae, order, seasonal_order, save_plot_path

# Grid Search Loop for Products and Seeds
This cell runs SARIMAX for each product and seed, saving results and plots.

In [ ]:
import os
import numpy as np
import pandas as pd

# Assumes you already defined:
# - generate_autoarima_candidates(...)
# - sarimax_recursive_forecast(...)
# - df, train_size, val_size, forecast_horizon
# - DATE_COL, TARGET_COL, EXOG_COLS

results = []
all_candidates = []
os.makedirs("grid_search_plots/sarimax", exist_ok=True)

# --- choose which products to run ---
products = df[["item_id", "store_id"]].drop_duplicates().values
target_products = [921558, 26008]
if target_products:
    products = df[df["item_id"].isin(target_products)][["item_id", "store_id"]].drop_duplicates().values

# --- build candidates per (item, store) and evaluate them ---
for item_id, store_id in products:
    df_product = df[(df["item_id"] == item_id) & (df["store_id"] == store_id)].copy()
    df_product = df_product.reset_index(drop=True)
    df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
    df_product = df_product.sort_values(DATE_COL).reset_index(drop=True)

    # 1) Generate AutoARIMA candidate list (orders + seasonal_orders)
    cand_df = generate_autoarima_candidates(
        df_product=df_product,
        date_col=DATE_COL,
        target_col=TARGET_COL,
        exog_cols=EXOG_COLS,
        train_size=train_size,
        val_size=val_size,
        forecast_window=forecast_horizon,
        seasonal_periods=(0, 7),   # daily retail: (0,7). monthly: (0,12)
        top_k=5,
        n_random_fits=80,
        random_state=42,
    )

    if cand_df is None or len(cand_df) == 0:
        print(f"[WARN] No candidates produced for item={item_id}, store={store_id}")
        continue

    # Optional: ensure tuples (sometimes they come back as lists/strings depending on earlier processing)
    def _to_tuple(x):
        if isinstance(x, tuple):
            return x
        if isinstance(x, list):
            return tuple(x)
        if isinstance(x, str):
            return tuple(eval(x))  # only if you KNOW it's safe in your pipeline
        return tuple(x)

    # 2) Evaluate each candidate using your SARIMAX procedure
    for i, row in cand_df.reset_index(drop=True).iterrows():
        order = _to_tuple(row["order"])
        seasonal_order = _to_tuple(row["seasonal_order"])

        plot_filename = (
            f"grid_search_plots/sarimax/"
            f"sarimax_item{item_id}_store{store_id}_cand{i}_"
            f"order{order}_seasonal{seasonal_order}.png"
        )

        forecast, rmse, mae, used_order, used_seasonal_order, plot_path = sarimax_recursive_forecast(
            df=df_product,
            target=TARGET_COL,
            exog_cols=EXOG_COLS,
            item_id=item_id,
            store_id=store_id,
            train_size=train_size,
            val_size=val_size,
            forecast_window=forecast_horizon,
            save_plot_path=plot_filename,
            order=order,
            seasonal_order=seasonal_order
        )

        results.append({
            "item_id": item_id,
            "store_id": store_id,
            "cand_rank": i,                 # index in the candidate list
            "m": int(row.get("m", 0)),       # seasonal period used when searching
            "order": used_order,
            "seasonal_order": used_seasonal_order,
            "rmse": rmse,
            "mae": mae,
            "ic": row.get("ic", np.nan),     # info criterion from candidate gen (if present)
            "aic": row.get("aic", np.nan),
            "bic": row.get("bic", np.nan),
            "aicc": row.get("aicc", np.nan),
            "plot_path": plot_filename
        })

# --- save results ---
results_df = pd.DataFrame(results)
results_df.to_csv("sarimax_autoarima_candidates_results.csv", index=False)

# Optional: best per (item, store)
best_df = (
    results_df.sort_values(["item_id", "store_id", "rmse", "mae"])
              .groupby(["item_id", "store_id"], as_index=False)
              .first()
)
best_df.to_csv("sarimax_autoarima_candidates_best.csv", index=False)

print("Saved:")
print(" - sarimax_autoarima_candidates_results.csv")
print(" - sarimax_autoarima_candidates_best.csv")
print(best_df)
candidates_df = pd.concat(all_candidates, ignore_index=True)
candidates_df.to_csv("autoarima_candidates.csv", index=False)

print(candidates_df)

In [7]:

results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Example grid search orders (can be expanded)
orders = [(1,1,1), (2,1,2)]
seasonal_orders = [(0,0,0,0), (1,0,1,12)]

for item_id, store_id in products:
    df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
    df_product = df_product.reset_index(drop=True)
    df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
    df_product = df_product.sort_values(DATE_COL)
    df_product = df_product.reset_index(drop=True)
    for order in orders:
        for seasonal_order in seasonal_orders:
            plot_filename = f'grid_search_plots/sarimax/sarimax_recursive_forecast_item{item_id}_store{store_id}_order{order}_seasonal{seasonal_order}.png'
            forecast, rmse, mae, used_order, used_seasonal_order, plot_path = sarimax_recursive_forecast(
                df=df_product,
                target=TARGET_COL,
                exog_cols=EXOG_COLS,
                item_id=item_id,
                store_id=store_id,
                train_size=train_size,
                val_size=val_size,
                forecast_window=forecast_horizon,
                save_plot_path=plot_filename,
                order=order,
                seasonal_order=seasonal_order
            )
            results.append({
                'item_id': item_id,
                'store_id': store_id,
                'order': used_order,
                'seasonal_order': used_seasonal_order,
                'rmse': rmse,
                'mae': mae,
                'plot_path': plot_filename
            })
results_df = pd.DataFrame(results)
results_df.to_csv('sarimax_grid_search_results.csv', index=False)

c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelih

KeyboardInterrupt: 